In [1]:
import os
import matplotlib.pylab as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
from scipy.stats import norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import matplotlib as mpl

random.seed(4)  
np.random.seed(4)

path = os.path.join(os.getcwd(), "15road_network_features.csv")
save_name = os.path.join(os.getcwd(), "road_network_classification_results.csv")
data = pd.read_csv(path)

columns_to_normalize = data.columns[2:]
data[columns_to_normalize] = data[columns_to_normalize].astype('float64')

df_name_features = [
    'number_of_branches', 'avg_branch_length', 'edge_density',
    'mean_betweenness_centrality', 'diameter', 'average_path_length',
    'assortativity', 'spectral_entropy', 'circuity',
    'mass_quartile_1', 'mass_quartile_2', 'mass_quartile_3', 'mass_quartile_4',
    'circular_std_dev', 'weighted_entropy', 'orientation_order',
    'fractal_dimension', 'algebraic_connectivity', 'avg_bending_energy'
]

df_grid = data[data['type'] == 'grid']  # Grid: 0
df_organic = data[data['type'] == 'organic']  # Organic: 1
df_hybrid = data[data['type'] == 'hybrid']  # Hybrid: 2

label_counts = {
    'Grid (0)': len(df_grid),
    'Organic (1)': len(df_organic),
    'Hybrid (2)': len(df_hybrid)
}
print("Class counts:", label_counts)

for label, count in label_counts.items():
    if count == 0:
        raise ValueError(f"No samples found for class {label}. Please ensure all classes have data in the CSV.")

df_feat_grid = df_grid.drop(['region_name', 'type'], axis=1).to_numpy()
df_feat_organic = df_organic.drop(['region_name', 'type'], axis=1).to_numpy()
df_feat_hybrid = df_hybrid.drop(['region_name', 'type'], axis=1).to_numpy()

df_lab_grid = np.zeros(len(df_grid), dtype=int)
df_lab_organic = np.ones(len(df_organic), dtype=int)
df_lab_hybrid = np.full(len(df_hybrid), 2, dtype=int)

df_feat = np.concatenate((df_feat_grid, df_feat_organic, df_feat_hybrid), axis=0)
df_lab = np.concatenate((df_lab_grid, df_lab_organic, df_lab_hybrid), axis=0)

rf_classifier = RandomForestClassifier(n_estimators=100,
                                       max_depth=20,
                                       criterion='gini',
                                       random_state=0)

classifiers = [rf_classifier]
names = ["RandomForest"]

road_results = [["Classifier", "Accuracy on Test Set", "Std"]]
results = [road_results]

num_iter = 10
all_importances = []

# Dictionary to store accuracies per location
location_accuracies = {loc: [] for loc in ['grid', 'organic', 'hybrid', 'overall']}

for i in range(num_iter):
    performance_scores = []

    for ds_cnt, ds in enumerate([[df_feat, df_lab]]):
        X, y = ds
        X = StandardScaler().fit_transform(X)

        random_state_1 = random.randint(0, 12)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state_1, stratify=y)

        for name, clf in zip(names, classifiers):
            print(f"Classifier: {name}, Iteration: {i+1}")
            clf.fit(X_train, y_train)

            # Aggregate the feature importances
            feature_importances = clf.feature_importances_
            all_importances.append(feature_importances * 100)

            # testing
            y_pred_proba = clf.predict_proba(X_test)
            pred_test = clf.predict(X_test)

            # Evaluate the performance metrics for the test dataset
            score = clf.score(X_test, y_test)
            conf = confusion_matrix(y_test, pred_test)

            report = classification_report(y_test, pred_test, output_dict=True)
            print("Classification report for iteration", i+1)
            print(classification_report(y_test, pred_test))

            # Calculate accuracy per location
            for loc, label in zip(['grid', 'organic', 'hybrid'], [0, 1, 2]):
                mask = (y_test == label)
                loc_accuracy = accuracy_score(y_test[mask], pred_test[mask]) if np.sum(mask) > 0 else 0
                location_accuracies[loc].append(loc_accuracy)

            # Overall accuracy
            overall_accuracy = accuracy_score(y_test, pred_test)
            location_accuracies['overall'].append(overall_accuracy)

            accuracy = overall_accuracy
            performance_scores.append(accuracy)

        average_performance = sum(performance_scores) / len(performance_scores)
        std_dev_performance = np.std(performance_scores)
        results[ds_cnt].append([name, average_performance, std_dev_performance])

# Aggregate and print per-location accuracies
print("\nPer-Location and Overall Accuracy Results (Mean ± Std):")
location_summary = {}
for loc in ['grid', 'organic', 'hybrid', 'overall']:
    mean_acc = np.mean(location_accuracies[loc]) if location_accuracies[loc] else 0
    std_acc = np.std(location_accuracies[loc]) if location_accuracies[loc] else 0
    location_summary[loc] = f"{mean_acc:.2f} ± {std_acc:.2f}"
print(" | ".join([f"{loc.upper()}" for loc in ['grid', 'organic', 'hybrid', 'overall']]))
print(" | ".join([location_summary[loc] for loc in ['grid', 'organic', 'hybrid', 'overall']]))

all_importances = np.array(all_importances)
final_mean_importances = np.mean(all_importances, axis=0)
final_std_importances = np.std(all_importances, axis=0)
sorted_indices = np.argsort(final_mean_importances)[::-1]
sorted_feature_names = np.array(df_name_features)[sorted_indices]
sorted_mean_importances = final_mean_importances[sorted_indices]
sorted_std_importances = final_std_importances[sorted_indices]

# For plot, reverse to have high at top
plot_feature_names = sorted_feature_names[::-1]
plot_mean_importances = sorted_mean_importances[::-1]
plot_std_importances = sorted_std_importances[::-1]

fig, ax = plt.subplots(figsize=(12, 8))
forest_importances = pd.Series(plot_mean_importances, index=plot_feature_names)
forest_importances.plot.barh(ax=ax, xerr=plot_std_importances)
ax.set_title("Aggregated Feature Importances Across Iterations (Mean ± Std)", fontsize=22)
ax.set_xlabel("MDI (%)", fontsize=16)
ax.tick_params(axis='y', labelsize=16)
ax.tick_params(axis='x', labelsize=16)
sns.despine()
fig.tight_layout()
plt.show()

print("Aggregated Feature Importances Across Iterations:")
for name, imp, std in zip(sorted_feature_names, sorted_mean_importances, sorted_std_importances):
    print(f"{name}: {imp:.1f}% ± {std:.1f}")

importances_df = pd.DataFrame({
    'Feature': sorted_feature_names,
    'Mean Importance (%)': np.round(sorted_mean_importances, 1),
    'Std Importance': np.round(sorted_std_importances, 1)
})
importances_save_name = os.path.join(os.getcwd(), "road_network_feature_importances.csv")
importances_df.to_csv(importances_save_name, index=False)
print(f"Aggregated feature importances saved to: {importances_save_name}")

results_df = pd.DataFrame(results[0][1:], columns=results[0][0])
results_df.to_csv(save_name, index=False)
print(f"Results saved to: {save_name}")

KeyboardInterrupt: 